This notebook is adapted from [Dataflowr Module's 2 online ressource](https://github.com/dataflowr/notebooks/tree/master/Module2).

To access this notebook on colab: https://colab.research.google.com/drive/1i1Y7DQKPvx27w7rWFL_5mTT5t0NWlOcK?usp=sharing.

# PyTorch tensors 101

The goal of this notebook is to manipulate [PyTorch](https://docs.pytorch.org/docs/stable/index.html) that allows to handle ```tensors```, which are used to encode the signal to process, but also the internal states and parameters of models.

Key ideas for PyTorch tensors are:
- Creation (`tensor`, `zeros`, `ones`, `arange`, `linspace`, `randn`, `eye`, `empty`)
- Attributes (`dtype`, `shape`, `device`)
- Indexing/slicing, reshaping (`view`, `reshape`, `unsqueeze`, `squeeze`)
- Broadcasting
- In-place operations
- Numpy bridge
- Moving tensors to GPUs (`.to(device)`, `.cpu()`, `.cuda()`)

In [ ]:
import matplotlib.pyplot as plt     # For plotting
%matplotlib inline
import torch                        # For tensor computations and deep learning
import numpy as np                  # For numerical operations on arrays

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# If you want to install PyTorch: visit the bottom page of the website https://pytorch.org.

### Creation and attributes

> **Q1.** Using the [`empty`](https://docs.pytorch.org/docs/stable/generated/torch.empty.html) function from Torch, construct an uninitialized 3x5 matrix. Check its default datatype and shape using the [`dtype`](https://docs.pytorch.org/docs/stable/tensor_attributes.html) and `shape` attributes, and print the values of the tensors. Comment.

In [ ]:
# YOUR CODE HERE

> **Q2.** Similarly, create a tensor of size 3x5 containing random values from a Gaussian $\mathcal{N}(0, 1)$. To do so, have a look at the [`randn`](https://docs.pytorch.org/docs/main/generated/torch.randn.html) function. Run the cell multiple times. Comment.

In [3]:
# YOUR CODE HERE

One way to enforce reproducibility when dealing with randomness in PyTorch is fixing the seed. This can be achieved using the `torch.manual_seed()` function of a generator.

In [ ]:
g = torch.Generator().manual_seed(42)
x = torch.randn(3, 5, generator=g)
print(x)

Here are some popular functions to initialize torch tensors:

In [ ]:
z = torch.zeros((2, 3, 3))
o = torch.ones((2, 3, 3))
e = torch.eye(3, 3)
l = torch.linspace(-1, 1, 11)
a = torch.arange(-1, 1, 0.2)

print("zeros:\n", z)
print("ones:\n", o)
print("eye:\n", e)
print("linspace:\n", l)
print("arange:\n", a)

### Indexing, reshaping, and slicing

To access an element of a tensor, we can simply specify the index in each dimension (starting from 0). For instance `x[2, 3]` access the element in the 3rd row, 4th column of $x$ and gives it back in a scalar-tensor.

In [ ]:
x = torch.randn(4, 4)
print(x)
print(x[2, 3])
print(x[2][3])      # x[2][3] is equivalent to x[2, 3]

In [ ]:
a = torch.arange(24)
print(a)

> **Q3.** $a$ is a tensor of size (24). [`Reshape`](https://docs.pytorch.org/docs/stable/generated/torch.reshape.html) it into a 2x3x4 tensor and check its shape. Note that reshape may copy the tensor in memory.

In [6]:
# YOUR CODE HERE

Another way to reshape a tensor is using [`view`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html#torch.Tensor.view) function.

In [ ]:
b2 = a.view(2, 3, 4)
print(b2)       # Same as b

However, it requires the tensor to be contiguous in memory. After some operations (like transpose or permute), the tensor is often non-contiguous so `view` will fail.

In [ ]:
y = b2.T     # likely non-contiguous
y2 = y.view(24)             # .view fails

> **Q4.** Slice the tensor $a$ to get a new one of shape (3, 2) containing the values: `[[1, 2], [5, 6], [9, 10]]`.

In [ ]:
# YOUR CODE

### [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)

Broadcasting automagically expands dimensions by replicating coefficients, when it is necessary to perform operations.

1. If one of the tensors has fewer dimensions than the other, it is reshaped by adding as many dimensions of size 1 as necessary in the front; then
2. For every mismatch, if one of the two tensor is of size one, it is expanded along this axis by replicating  coefficients.

If there is a tensor size mismatch for one of the dimension and neither of them is one, the operation fails.

In [237]:
v = torch.tensor([1.0, 2.0, 3.0])

> **Q5.** Create a random matrix of size 5x3. Then add $v$ to it. What happens to the resulting tensor? Explain.

In [ ]:
# YOUR CODE HERE

> **Q6.** Similarly, explain how the broadcasting worked in the code below.

In [240]:
A = torch.arange(1, 5).unsqueeze(1)
print(A.shape)
B = torch.tensor([[5., -5., 5., -5., 5.]])
print(B.shape)
C = A + B
print(C)

torch.Size([4, 1])
torch.Size([1, 5])
tensor([[ 6., -4.,  6., -4.,  6.],
        [ 7., -3.,  7., -3.,  7.],
        [ 8., -2.,  8., -2.,  8.],
        [ 9., -1.,  9., -1.,  9.]])


### In-place modifications

In-place operations directly modify the content of a tensor. These operations have a suffix `_`. For example: `x.copy_(y)`, `x.add_(y)`, `x.t_()`, `x.fill_(y)` will all change `x`. They can help saving some memory but can cause problems when computing gradients.

In [ ]:
x = torch.randn(5, 3)
b = torch.ones(5, 3)

print(x+b)
print(x)

In [ ]:
x.add_(b)
print(x)        # x has changed

In [ ]:
x.t_()
print(x)

### Bridge to numpy

Torch tensors can be converted to `numpy.ndarray` using the [`torch.Tensor.numpy`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.numpy.html) function which can be used as

In [ ]:
x = torch.randn(5, 3)
y = x.numpy()
print(y)
print(type(y))

Similarly, a `numpy.ndarray` can be transformed into a `torch.tensor`using the [`torch.from_numpy()`](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html) function.

In [ ]:
a = np.ones(5)
b = torch.from_numpy(a)
print(a.dtype)
print(b)

In [ ]:
xr = torch.randn(3, 5)
print(xr.dtype, xr)

Note: Be careful with types!

In [ ]:
b1 = torch.ones(5, dtype=torch.float64)
b2 = torch.ones(5, dtype=torch.long)
r = torch.randn(3, 5, generator=torch.Generator().manual_seed(42))
x1 = r + b1
x2 = r + b2

print(x1)
print(x2)
print(b1)
print(b2)
print(x1 == x2)     # Falses because different dtypes

### Shared memory

Also be careful, changing the torch tensor modify the numpy array and vice-versa...

This is explained in the PyTorch documentation [here](https://pytorch.org/docs/stable/torch.html#torch.from_numpy):
The returned tensor by `torch.from_numpy` and ndarray share the same memory. Modifications to the tensor will be reflected in the ndarray and vice versa. 

In [ ]:
a = np.ones(5)
b = torch.from_numpy(a)
print(b)

In [ ]:
a[2] = 0
print(b)    # b has changed while we only modified a

In [ ]:
b[3] = 5
print(a)    # a has changed while we only modified b

### Devices and cuda

Tensors can be loaded either on CPUs or on GPUs. To check whether GPUs are available, we can call `torch.cuda.is_available()` returning a boolean.

In [ ]:
torch.cuda.is_available()

In [209]:
device = torch.device('cpu')   # Use this to run on CPU
# device = torch.device('cuda')   # Use this to run on GPU

In [ ]:
x = torch.ones(10)
print(x.device)  # Should show "cpu" (default device)

To load something on a specific device, we should create the tensor and precise the device argument.

In [ ]:
x = torch.ones(10, device=device)
print(x.device)  # Should show "cpu" or "cuda" depending on the device

In [212]:
# let us run this cell only if CUDA is available
# We will use ``torch.device`` objects to move tensors in and out of GPU
if torch.cuda.is_available():
    y = torch.ones_like(x, device=device)  # directly create a tensor on GPU
    x = x.to(device)                       # or just use strings ``.to("cuda")``
    z = x + y
    print(z,z.type())
    print(z.to("cpu", torch.double))       # ``.to`` can also change dtype together!

In [213]:
x = torch.randn(1)
x = x.to(device)

In [ ]:
x.device

In [ ]:
# the following line is only useful if CUDA is available
x = x.data
print(x)
print(x.item())
print(x.cpu().numpy())  # move to CPU first, then convert to numpy

## Simple interfaces to standard image data-bases

An example with the [CIFAR10](https://pytorch.org/docs/stable/torchvision/datasets.html#torchvision.datasets.CIFAR10) dataset.

In [ ]:
import torchvision

data_dir = 'content/data'

cifar = torchvision.datasets.CIFAR10(data_dir, train = True, download = True)
cifar.data.shape

Documentation about the [`permute`](https://pytorch.org/docs/stable/tensors.html#torch.Tensor.permute) operation.

In [ ]:
x = torch.from_numpy(cifar.data).permute(0,3,1,2).float()
x = x / 255
print(x.type(), x.size(), x.min().item(), x.max().item())

Documentation about the [`narrow(input, dim, start, length)`](https://pytorch.org/docs/stable/torch.html#torch.narrow) operation.

In [ ]:
# Narrows to the first images, converts to float
x = torch.narrow(x, 0, 0, 48)

In [ ]:
x.shape

In [ ]:
# Showing images
def show(img):
    npimg = img.numpy()
    plt.figure(figsize=(20,10))
    plt.imshow(np.transpose(npimg, (1,2,0)), interpolation='nearest')
    
show(torchvision.utils.make_grid(x, nrow = 12))

In [ ]:
# Kills the green and blue channels
x.narrow(1, 1, 2).fill_(0)
show(torchvision.utils.make_grid(x, nrow = 12))